# Build a Knowledge Graph for Your OSPF Network — the guided version

*A warm, step-by-step lab for network engineers. You bring the OSPF. We bring the graph.*

You already run the biggest graph database in your building. It is called **OSPF**.
Your routers are **nodes**. Your adjacencies are **edges**. `SPF` is a **traversal** —
Dijkstra walking that graph to pick shortest paths. You have been operating a graph
engine your whole career; nobody just called it that.

A **knowledge graph** is that *same machine* — except **you** choose the nodes. And that
one freedom changes everything, because it lets you add the single node OSPF was never
designed to hold: the **business service** that dies when a route disappears.

By the end of this notebook you will have built, from an empty page, a small graph that
answers the question no `show` command can:

> *"abr-01 just lost its backbone link. Which business service is now at risk?"*

and watched it print **`Order API`** — a fact that lives in nobody's routing table.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is
just **structured facts** (things, and the named links between them) plus **queries** that
walk those links. Everything is **deterministic and inspectable** — run it twice, get the
same answer, and you can point at every node that produced it. If you can read a routing
table, you can read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python +
networkx + matplotlib** (both already installed in Colab). No database, no Docker, no
credentials. Press *Runtime → Run all*, or run each cell in order with **Shift+Enter**.

## The whole vocabulary — five words, each one an OSPF thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five
ideas, repeated. And notice: you don't have to learn what any of them *mean* — you only
have to learn which OSPF thing each one maps onto.

| Graph word | Plain meaning | The OSPF thing it already is |
|---|---|---|
| **node** | a thing | a router, an area, a prefix, a service |
| **edge** | a named, directed link between two nodes | an adjacency, "this ABR connects that area" |
| **label** | a node's *type* | `Area`, `Router`, `Prefix` — like "this is an ABR vs an ASBR" |
| **property** | a fact stored *on* a node or edge | `state='down'`, `area_id='0.0.0.0'`, `criticality='critical'` |
| **traversal** | walking edges to answer a question | exactly what SPF does — you'll do it by hand, on purpose |

That's it. Hold those five words and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an
edge, not just a node.** "The link is down" is not really a property of the router or of
the area — it is a property of the *adjacency between them*. OSPF knows this in its bones
(interface state, neighbor state). In a graph you write it down literally: the failure
lives **on the edge**. Keep an eye out for that moment — it is where a topology diagram
turns into something you can *query*.

## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in
memory. We will use a **`DiGraph`** — a *directed* graph, where every edge has a
direction, an arrow. That matters to us: `abr-01 → Area 0` ("abr-01 connects to Area 0")
is a different statement from `Area 0 → abr-01`. OSPF is full of directional truth — who
originates an LSA, which way a summary floods — so directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print("Empty graph ready:", G)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty
`G` is your blank whiteboard. Every step from here just adds nodes and edges to it.

## Step 1 — Areas: the hierarchy, as nodes

**Theory.** You already know *why* OSPF has areas: to keep the link-state database and SPF
from melting under one giant flat topology. **Area 0 is the backbone** — the spine every
other area must touch to reach the rest of the domain. That hierarchy is the first thing
worth writing into our graph, because most of the interesting failures happen *at area
boundaries*.

In graph terms, each area becomes a **node** with a **label** of `Area` and a couple of
**properties**: its `area_id` and its `area_type`. We'll model the classic three-area
design from the field: **Area 0** (backbone), **Area 10** (a normal area where our app
lives), and **Area 20** (a stub area facing a partner).

Notice we are *not* dumping the LSDB in here. A node per area — not a node per LSA entry.
We store the *shape* of the design, and we'll come back to that principle by name later.

In [ ]:
# add_node(name, **properties). The name is just a unique handle; label & area_id are facts on it.
G.add_node("Area 0",  label="Area", area_id="0.0.0.0",  area_type="backbone")
G.add_node("Area 10", label="Area", area_id="0.0.0.10", area_type="normal")
G.add_node("Area 20", label="Area", area_id="0.0.0.20", area_type="stub")

print(G.number_of_nodes(), "nodes so far:", list(G.nodes))
print("Facts on Area 0:", G.nodes["Area 0"])

**Checkpoint.** Three nodes: `Area 0`, `Area 10`, `Area 20`, and the facts on Area 0 show
`area_id='0.0.0.0'` and `area_type='backbone'`. You just wrote your OSPF hierarchy into a
graph — no edges yet, just the actors.

## Step 2 — Routers and their roles

**Theory.** A router's *role* is not the same as the router. The same box can be an **ABR**
(sits on the boundary between area 0 and another area) and, on a different day or a
different interface, an **ASBR** (imports reachability from another domain). You already
think this way — "is this the ABR into Area 10?" is a question about a *role*, not a serial
number.

So in the graph, `role` is a **property** on the router node. We add two boundary routers:

- **abr-01** — the Area Border Router that stitches Area 10 to the backbone.
- **asbr-01** — the Autonomous System Boundary Router that holds Area 20 and the backbone.
  (Strictly, bordering the backbone and another area makes it an ABR too; the ASBR job —
  importing routes from *outside* OSPF — is off-screen in this small model, so here it just
  keeps Area 20 healthy.)

Here is the design tension we're deliberately baking in: **Area 10 hangs off a single
ABR.** In a real network you'd want two, precisely so one uplink loss can't isolate the
area. We model one on purpose — so the failure, and its blast radius, shows up cleanly in a
single edge you can point at.

In [ ]:
G.add_node("abr-01",  label="Router", role="ABR",  router_id="10.255.3.1")
G.add_node("asbr-01", label="Router", role="ASBR", router_id="10.255.3.2")

for n, d in G.nodes(data=True):
    if d.get("label") == "Router":
        print(f'{n}: role={d["role"]}, router_id={d["router_id"]}')

**Checkpoint.** Two routers print with their roles: `abr-01` is an `ABR`, `asbr-01` is an
`ASBR`. They're floating unconnected for now — the next step wires them to the areas, and
that wire is where the story turns.

## Step 3 — Adjacencies: the link carries the state (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. When a backbone uplink drops, *where*
does that fact belong? Not on the router (the router is fine). Not on the area (the area is
fine). It belongs on the **relationship between them** — the adjacency. OSPF already agrees
with you: interface state and neighbor state describe the *link*, not the endpoints.

So we add **edges** with a `rel` of `CONNECTS`, and we hang a `state` **property** right on
each edge. Read `abr-01 --CONNECTS(down)--> Area 0` literally: *"abr-01 connects to Area 0,
and that connection is down."*

The wiring, mirroring the field design:

- `abr-01 → Area 0` is **down** — the broken backbone uplink. This one edge is the incident.
- `abr-01 → Area 10` is **up** — Area 10 is still reachable *from* abr-01 locally.
- `asbr-01 → Area 0` is **up** and `asbr-01 → Area 20` is **up** — the backbone itself stays
  alive, because asbr-01 is still sitting in it. (A backbone with a live router in it is,
  by definition, not isolated. What gets cut off is **Area 10**, because abr-01 is its only door.)

In [ ]:
# add_edge(source, target, **properties). The failure is a property ON the edge.
G.add_edge("abr-01",  "Area 0",  rel="CONNECTS", state="down")   # <-- the whole incident
G.add_edge("abr-01",  "Area 10", rel="CONNECTS", state="up")
G.add_edge("asbr-01", "Area 0",  rel="CONNECTS", state="up")
G.add_edge("asbr-01", "Area 20", rel="CONNECTS", state="up")

for u, v, d in G.edges(data=True):
    print(f'{u} --{d["rel"]}({d["state"]})--> {v}')

**Checkpoint.** Four `CONNECTS` edges print, and exactly one reads `down`: the
`abr-01 --CONNECTS(down)--> Area 0` line. That single edge, with a property on it, is the
3 AM event — sitting in your graph as a fact you can now query against.

## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the
picture. We'll colour nodes by their **label** (areas one colour, routers another) and draw
the one **down** edge as a dashed red arrow so the failure jumps out. This is the same
instinct as a topology diagram — except this diagram is generated *from the data*, so it can
never drift out of sync with the truth.

In [ ]:
def draw(G, title="OSPF knowledge graph"):
    colors = {"Area": "#0f7f74", "Router": "#3aa0ff", "Prefix": "#9aa5ad",
              "Service": "#c8702a", "LSA": "#8a6fb0"}
    node_colors = [colors.get(G.nodes[n].get("label"), "#cccccc") for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=9)

    down  = [(u, v) for u, v, d in G.edges(data=True) if d.get("state") == "down"]
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color="#8894a0")
    nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16,
                           edge_color="#d34b3f", width=2.0, style="dashed")
    nx.draw_networkx_edge_labels(G, pos, font_size=7,
        edge_labels={(u, v): d.get("rel", "") for u, v, d in G.edges(data=True)})

    plt.axis("off"); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the areas (teal) and routers (blue) with `CONNECTS`
arrows between them, and one **dashed red** arrow from `abr-01` to `Area 0` — the down link.
This is still just a topology. It becomes a *knowledge* graph in the next step, when we add
the things a topology diagram has never carried: the prefix, and the business behind it.

## Step 4 — Prefix lineage: give a route its provenance

**Theory.** On a topology diagram a prefix is just a label floating near an area. In a
knowledge graph a prefix gets **provenance** — a traceable story of *where it comes from*:
which area it originates in, and which LSA carries it. That story is exactly what evaporates
during an incident, scattered across the LSDB, the RIB, and someone's memory.

Two new nodes and their edges:

- a **Prefix** node, `10.30.10.0/24`, the application subnet that lives in Area 10;
- an **LSA** node — specifically the **Type 3 summary** that `abr-01` floods into the
  backbone so *other* areas can reach that prefix. We tag it `status='withdrawn'`, because
  the moment abr-01 loses the backbone it stops originating that summary (RFC 2328 §16) and
  flushes it.

The **stop-and-think rule** you're using here, without me naming it yet: store *summarized
operational truth and dependency lineage* — a node for the prefix and a node for the LSA
that matters — and leave the 40,000-entry LSDB where it lives, in the router. The graph
holds the *shape* of the dependency, not the telemetry firehose.

(Nuance a network engineer will hold me to: inside Area 10 that prefix is an intra-area
route on Type 1/2 LSAs, and hosts *in* Area 10 keep reaching it. What the withdrawn Type 3
kills is *inter-area* reachability — every other area loses the prefix, and Area 10 loses
the rest of the domain. That's the failure we're about to make queryable.)

In [ ]:
G.add_node("10.30.10.0/24", label="Prefix", visibility="lost_inter_area")
G.add_node("Type 3 Summary (Area 10)", label="LSA", lsa_type=3, status="withdrawn")

G.add_edge("10.30.10.0/24", "Area 10",                    rel="ORIGINATES_IN")
G.add_edge("10.30.10.0/24", "Type 3 Summary (Area 10)",   rel="ADVERTISED_BY")

print("Prefix facts:", G.nodes["10.30.10.0/24"])
print("Edges out of the prefix:")
for u, v, d in G.out_edges("10.30.10.0/24", data=True):
    print(f'  {u} --{d["rel"]}--> {v}')

**Checkpoint.** The prefix now has two outgoing edges: `ORIGINATES_IN → Area 10` and
`ADVERTISED_BY → Type 3 Summary (Area 10)`. The route has a *story* now. But a route with a
story still isn't a business fact — that's the final, missing node, next.

## Step 5 — The business service: the node OSPF never had

**Theory.** This is the node your routers were never designed to hold, and the reason the
whole lab exists. OSPF knows adjacencies, LSAs, costs, prefixes. It has never once known
that `10.30.10.0/24` is where the **Order API** lives, and that when that prefix loses
inter-area reachability, **revenue stops**.

So we add a **Service** node, `Order API`, with a `criticality` property, and we wire it to
the prefix with a `SUPPORTS` edge: *"this prefix supports that service."* One edge — and now
a routing fact and a business fact are welded together in a place you can query. No `show`
command holds this link. It has never lived anywhere except tribal knowledge. You're about
to make it a first-class, walkable fact.

In [ ]:
G.add_node("Order API", label="Service", criticality="critical")
G.add_edge("10.30.10.0/24", "Order API", rel="SUPPORTS")

print("Graph now has", G.number_of_nodes(), "nodes and", G.number_of_edges(), "edges.")
print("The load-bearing link:", [f'{u} -SUPPORTS-> {v}'
      for u, v, d in G.edges(data=True) if d.get("rel") == "SUPPORTS"])
draw(G, title="OSPF knowledge graph — now with the business service")

**Checkpoint.** The graph has grown to include an orange `Service` node, `Order API`, joined
by a `SUPPORTS` edge to the prefix. In the redrawn picture you can *trace with your finger*:
down link → abr-01 → Area 10 → prefix → Order API. In the next step we make the computer
trace it for you.

## Step 6 — The 3 AM question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to
answer a question — the exact thing SPF does, except *you* decide the walk. Our walk answers:

> *"For any router whose **backbone** link is **down**, which business services are at risk?"*

The route the walk takes:

1. find a router whose `CONNECTS` edge **to the backbone** (area_id `0.0.0.0`) has `state='down'`;
2. hop to the **other** areas that router serves (its non-backbone `CONNECTS` edges);
3. from each area, hop to the **prefixes** that `ORIGINATES_IN` it;
4. from each prefix, hop to the **services** it `SUPPORTS`.

Every hop is one edge. Crucially, the query is **conditioned on link state** — flip the
backbone edge to `up` and step 1 finds nothing, so the whole walk returns nothing. That
responsiveness is the difference between a static diagram and a live model. Run it.

In [ ]:
def blast_radius(G):
    # Services put at risk by any router whose BACKBONE link is currently down.
    at_risk = []
    for rtr, area0, d in G.edges(data=True):
        # 1) a CONNECTS edge to the backbone (area_id 0.0.0.0) that is DOWN
        if d.get("rel") == "CONNECTS" and d.get("state") == "down" \
           and G.nodes[area0].get("area_id") == "0.0.0.0":
            # 2) the other, non-backbone areas this router serves
            for _, area, d2 in G.out_edges(rtr, data=True):
                if d2.get("rel") != "CONNECTS" or G.nodes[area].get("area_id") == "0.0.0.0":
                    continue
                # 3) prefixes that originate in that area
                for pfx, _, d3 in G.in_edges(area, data=True):
                    if d3.get("rel") != "ORIGINATES_IN":
                        continue
                    # 4) services those prefixes support
                    for _, svc, d4 in G.out_edges(pfx, data=True):
                        if d4.get("rel") == "SUPPORTS":
                            at_risk.append((rtr, area, pfx, svc))
    return at_risk

hits = blast_radius(G)
for rtr, area, pfx, svc in hits:
    print(f"{rtr} backbone DOWN  ->  {area}  ->  {pfx}  ->  AT RISK: {svc}")
if not hits:
    print("nothing at risk")

The router only ever told you an adjacency dropped — one line in a log. Your graph just told
you the **Order API is at risk**, and showed its work: the whole path from the down link to
the revenue. You got that answer because *you* added the one node OSPF never had. That is the
entire pitch of a knowledge graph, and you just built it from an empty page.

## Step 7 — What-if: repair the link, then break it again

**Theory.** Because the failure lives on an edge as a plain property, you can *change* it and
re-ask — running "what if this fails?" (or "what if I fix it?") on a **model**, safely, with
no one paged and no maintenance window. Flip the backbone edge to `up`: the blast radius
clears. Flip it back to `down`: the Order API returns. This is the humble beginning of
pre-incident planning — you can ask the graph what *would* break before it does.

In [ ]:
# You can read and write an edge's properties directly by [source][target].
G["abr-01"]["Area 0"]["state"] = "up"       # repair the uplink
print("After repair:   ", blast_radius(G) or "nothing at risk")

G["abr-01"]["Area 0"]["state"] = "down"     # break it again
print("After re-break: ", [svc for *_, svc in blast_radius(G)])

**Checkpoint.** After the repair you should see `nothing at risk`; after re-breaking it,
`['Order API']` comes back. Same graph, same query — only the state on one edge changed. The
answer *responded*. That is what makes it a model rather than a drawing.

## Your turn #1 — a second service on the same prefix

Real prefixes rarely support just one thing. Suppose a **Payments Webhook** service also
rides on `10.30.10.0/24`. Add it, wire it with a `SUPPORTS` edge, and re-run `blast_radius`.
You should now see **two** services surface from the same down link — because one edge of
extra truth is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to
check.

In [ ]:
# TODO: add a "Payments Webhook" Service node (criticality="high"),
#       then a SUPPORTS edge from "10.30.10.0/24" to it.

# G.add_node(...)
# G.add_edge(...)

for rtr, area, pfx, svc in blast_radius(G):
    print(f"AT RISK: {svc}  (via {pfx} in {area})")

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Payments Webhook", label="Service", criticality="high")
G.add_edge("10.30.10.0/24", "Payments Webhook", rel="SUPPORTS")

for rtr, area, pfx, svc in blast_radius(G):
    print(f"AT RISK: {svc}  (via {pfx} in {area})")
```

Now both `Order API` **and** `Payments Webhook` come back from the one down backbone link.
The blast radius grew the instant you told the graph one more true thing — you didn't touch
the query at all.
</details>

In [ ]:
# (Solution, applied — so the rest of the notebook has both services in the graph.)
G.add_node("Payments Webhook", label="Service", criticality="high")
G.add_edge("10.30.10.0/24", "Payments Webhook", rel="SUPPORTS")
print("Services at risk now:", sorted({svc for *_, svc in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"An `Area` has an `area_id`. A `Prefix` `ORIGINATES_IN` an `Area`. A `Service` is
  `SUPPORTS`-ed by a `Prefix`."* — these are the **rules**: which node types exist, which
  edges are allowed, what shape a valid fact takes. That is the **schema**. Its fancy name is
  an **ontology** — and here's the friendly definition: *an ontology is the RFC of your graph,
  the agreed vocabulary written down as structure.* You already trust RFC 2328 to say what a
  valid OSPF adjacency is; an ontology does the same job for your graph.

- *"This particular router is `abr-01`, router-id `10.255.3.1`, and its backbone link is
  `down`."* — these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, an IP, or a
serial number, it is data (an instance), never schema.** `Router` is schema; `abr-01` is
data. `SUPPORTS` is schema; "this prefix supports Order API" is data. Keep the vocabulary
small and stable; let the instances be many. That single discipline is the difference between
a graph you can grow for years and a swamp you abandon in a month.

## A peek at the real thing (1/2) — the same areas in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are
exactly what you'd build in a real graph database like **Neo4j**, whose query language is
**Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been
drawing — `(node)-[:EDGE]->(node)`.

Here is **Step 1 (the areas)** as Cypher. `MERGE` means "find this node or create it if
missing" (so re-running is safe); `SET` assigns properties. This is *reference only* — you
don't run it here, it just shows you the same idea in the grown-up tool:

```cypher
MERGE (area0:OSPFArea:BackboneArea {id: 'area-0'})
SET   area0.area_id = '0.0.0.0',  area0.area_type = 'backbone';

MERGE (area10:OSPFArea:NormalArea {id: 'area-10'})
SET   area10.area_id = '0.0.0.10', area10.area_type = 'normal';

MERGE (abr:OSPFRouter:ABR {id: 'abr-01'})
SET   abr.router_id = '10.255.3.1';

// the failure, as a property on the relationship — same as our edge
MERGE (abr)-[:CONNECTS {state: 'down'}]->(area0);
```

See it? `(abr)-[:CONNECTS {state:'down'}]->(area0)` is character-for-character the same
statement as our `G.add_edge("abr-01", "Area 0", rel="CONNECTS", state="down")`. Same node,
same edge, same fact-on-the-edge — one just happens to live in a database that scales to
millions of nodes.

## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our `blast_radius` walk is a 20-line Python function. In Cypher, that same traversal is five
lines, because in a graph database you **draw the shape you're looking for** and let the
engine find every match — no manual loops:

```cypher
MATCH (abr:ABR)-[c:CONNECTS {state:'down'}]->(bb:OSPFArea {area_id:'0.0.0.0'})
MATCH (abr)-[:CONNECTS]->(area:OSPFArea)
WHERE area.area_id <> '0.0.0.0'
MATCH (prefix:Prefix)-[:ORIGINATES_IN]->(area)
MATCH (prefix)-[:SUPPORTS]->(svc:BusinessService)
RETURN abr.name AS abr, area.name AS area,
       collect(DISTINCT svc.name) AS services_at_risk;
```

Read the first line as a sentence: *"match an ABR whose CONNECTS-to-the-backbone edge is
down."* That's the same step 1 you wrote in Python — the pattern you `MATCH` **is** the
traversal. Run it against the real 3-area dataset in the companion Neo4j lab and `abr-01`
comes back with `Order API` **among** the services at risk (you'll also see `Fraud Detection
API`, another Area 10 service riding a prefix hidden by the same failure). Our tiny in-memory
graph models just the one service to keep the story clean. Different engine; identical thinking. If
you can read the Python above, you can read the Cypher — you already speak this language.

## Your turn #2 — make the query respond to a *different* failure

Area 20 (our stub, held by `asbr-01`) is perfectly healthy right now — its backbone link is
`up`, so nothing in it is at risk. Let's prove the model reacts to reality:

1. add a **Partner Portal** service on a new prefix `10.30.20.0/24` that `ORIGINATES_IN`
   `Area 20`;
2. re-run `blast_radius` — the Partner Portal should **not** appear (asbr-01's backbone is up);
3. now set `asbr-01 → Area 0` to `down` and re-run — the Partner Portal **appears**.

This is the whole point of Step 7, in your own hands: the answer follows the state. Fill in
the `# TODO`s, then run.

In [ ]:
# TODO 1: add Prefix "10.30.20.0/24" and Service "Partner Portal", then:
#   - ORIGINATES_IN edge: prefix -> "Area 20"
#   - SUPPORTS edge:      prefix -> "Partner Portal"

# G.add_node(...); G.add_node(...)
# G.add_edge(...); G.add_edge(...)

print("With asbr-01 backbone UP:  ", sorted({s for *_, s in blast_radius(G)}))

# TODO 2: break asbr-01's backbone link (G["asbr-01"]["Area 0"]["state"] = "down"),
#         then add a second print of blast_radius here to see Partner Portal appear.
#         (The solution below demonstrates the DOWN case in full.)

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("10.30.20.0/24", label="Prefix")
G.add_node("Partner Portal", label="Service", criticality="medium")
G.add_edge("10.30.20.0/24", "Area 20",        rel="ORIGINATES_IN")
G.add_edge("10.30.20.0/24", "Partner Portal", rel="SUPPORTS")

print("With asbr-01 backbone UP:  ", sorted({s for *_, s in blast_radius(G)}))
# -> Order API, Payments Webhook  (Partner Portal is safe: asbr-01 backbone is up)

G["asbr-01"]["Area 0"]["state"] = "down"
print("With asbr-01 backbone DOWN:", sorted({s for *_, s in blast_radius(G)}))
# -> now Partner Portal joins the list
```

Before you broke the link, Partner Portal was *in the graph* but *not at risk* — the query
correctly ignored a healthy path. The moment asbr-01 lost the backbone, the exact same query
surfaced it. You didn't teach the query about Area 20; you just told the graph the truth, and
the traversal did the rest.
</details>

In [ ]:
# (Solution, applied — leaves asbr-01 healthy again so later cells start from a clean state.)
G.add_node("10.30.20.0/24", label="Prefix")
G.add_node("Partner Portal", label="Service", criticality="medium")
G.add_edge("10.30.20.0/24", "Area 20",        rel="ORIGINATES_IN")
G.add_edge("10.30.20.0/24", "Partner Portal", rel="SUPPORTS")
G["asbr-01"]["Area 0"]["state"] = "down"
print("Partner failure surfaces:", sorted({s for *_, s in blast_radius(G)}))
G["asbr-01"]["Area 0"]["state"] = "up"   # restore health
print("Restored, at risk again:", sorted({s for *_, s in blast_radius(G)}))

## Make it yours — swap in *your* network

Now the moment it becomes your tool, not mine. Change the four values below to **one** area,
**one** ABR, **one** prefix, and **one** service *you* actually run. We add the down backbone
link for you, so your service shows up. Run it, and watch **your own service name** come back
from `blast_radius` — proof that the machine you built understands your network, not just the
demo's.

Keep it to the smallest slice that answers one real question. You can always add one more node
tomorrow — that's the whole promise of a graph you can grow.

In [ ]:
# --- change these four values to your network ---
MY_AREA    = "Campus Area 42"
MY_ROUTER  = "campus-abr-01"
MY_PREFIX  = "10.42.7.0/24"
MY_SERVICE = "Badge Access"
# ------------------------------------------------

MY_AREA_ID = "0.0.0.42"   # any non-zero area id works

G.add_node(MY_AREA,    label="Area",   area_id=MY_AREA_ID, area_type="normal")
G.add_node(MY_ROUTER,  label="Router", role="ABR")
G.add_node(MY_PREFIX,  label="Prefix")
G.add_node(MY_SERVICE, label="Service", criticality="critical")

G.add_edge(MY_ROUTER, "Area 0", rel="CONNECTS", state="down")   # your modelled failure
G.add_edge(MY_ROUTER, MY_AREA,  rel="CONNECTS", state="up")
G.add_edge(MY_PREFIX, MY_AREA,  rel="ORIGINATES_IN")
G.add_edge(MY_PREFIX, MY_SERVICE, rel="SUPPORTS")

print(f"If {MY_ROUTER}'s backbone link fails, these services are at risk:")
for rtr, area, pfx, svc in blast_radius(G):
    if rtr == MY_ROUTER:
        print(f"  AT RISK: {svc}   (via {pfx} in {area})")

**Checkpoint.** Your own service name — `Badge Access`, or whatever you typed — prints as *at
risk*. That is the moment the tool stopped being a tutorial and became yours. Every other
service you run is just four more lines away.

## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your monitoring dashboard.**

Areas, routers, roles, prefixes, LSAs, services, policies — the **nouns** you'd draw on a
whiteboard while arguing about a design — those belong in the graph. Interface counters, CPU
percentages, per-flow telemetry, the 40,000-row LSDB — those are the **numbers**; leave them
in the systems that already store them well, and let the graph *reference* them when it needs
to. The graph holds the *shape of the dependency*; your NMS holds the firehose. Keep that line
sharp and your graph stays queryable for years.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes,
  same edges, same 3 AM question — so the two Cypher peeks above become things you run.
- **Add a protocol.** The shapes generalize: a BGP session is an edge with state; an MPLS LSP
  is a path you traverse; a redistribution policy is a node other things point at. The same
  five words carry all of it.
- **Add the change layer.** Model a `ChangeEvent` node linked to what it touched, and "what
  changed right before this broke?" becomes one more traversal.

## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added
areas, routers, and a down link — a topology. Then you added a prefix's provenance and the one
node OSPF never had, the **business service** — and the topology became *knowledge*. Finally
you asked it the question no router can answer, and it printed **Order API**, then responded
correctly every time you changed the world underneath it.

The important idea was never OSPF, and never networkx. It's this: **operational truth has a
shape** — areas contain prefixes, prefixes ride LSAs, services depend on prefixes — and once
that shape is a graph, impact analysis stops being tribal knowledge and becomes a traversal.
A runbook is a hammer. A graph you can query is the whole toolbox.

You already ran the biggest graph in your building. Now you know how to build the one that
holds the part OSPF was never asked to remember. Go model one real service on your network,
and ask it what breaks.